In [2]:
import json
from pathlib import Path
import pandas as pd

In [3]:
PROJECT_ROOT = Path.cwd().parents[1]
DATA_PATH = PROJECT_ROOT / "files"

# Load annotated CVs
with open(DATA_PATH / "linkedin-cvs-annotated.json", "r", encoding="utf-8") as f:
    cvs = json.load(f)
len(cvs)

609

In [4]:
jobs = [job for cv in cvs for job in cv]
df = pd.DataFrame(jobs)

# Focus on ACTIVE as target
df_active = df[df["status"] == "ACTIVE"].copy()

def norm_text(s: str) -> str:
    if pd.isna(s):
        return ""
    return str(s).strip().lower()

df_active["position_norm"] = df_active["position"].apply(norm_text)

In [5]:
df_dep_v2 = pd.read_csv(DATA_PATH / "department-v2.csv")
df_dep_v2["text_norm"] = df_dep_v2["text"].apply(norm_text)

lookup_dep = (
    df_dep_v2.groupby("text_norm")["label"]
    .agg(lambda x: x.value_counts().idxmax())
    .to_dict()
)

In [6]:
df_active["department_pred_lookup"] = df_active["position_norm"].map(lookup_dep)

match_mask = df_active["department_pred_lookup"].notna()
match_rate = match_mask.mean()

# nur evaluieren, wenn truth im v2 label space liegt
v2_labels = set(df_dep_v2["label"].unique())
eval_mask = match_mask & df_active["department"].isin(v2_labels)

accuracy_on_matched = (df_active.loc[eval_mask, "department_pred_lookup"]
                       == df_active.loc[eval_mask, "department"]).mean()

print("Department lookup baseline")
print("Match rate (titles found in department-v2):", round(match_rate, 3))
print("Accuracy on matched (truth within v2 label set):", round(accuracy_on_matched, 3))
print("Coverage for evaluation (matched & truth in v2):", int(eval_mask.sum()), "/", len(df_active))

Department lookup baseline
Match rate (titles found in department-v2): 0.066
Accuracy on matched (truth within v2 label set): 0.951
Coverage for evaluation (matched & truth in v2): 41 / 623
